In [ ]:
!pip install -U transformers accelerate sentencepiece protobuf nltk
!pip install qwen-vl-utils

In [ ]:
import torch
from pathlib import Path
from tqdm.auto import tqdm
from IPython.display import FileLink, display
from transformers import AutoTokenizer
from huggingface_hub import hf_hub_download
from safetensors import safe_open

# Keep "llava" to reproduce this notebook's existing VLM tokenizer and weights.
VLM_CHOICE = "llava"

device = torch.device(
    "cuda:0" if torch.cuda.is_available() else "cpu"
)

print("Torch:", torch.__version__)
print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


VLM_REGISTRY = {
    "llava": {
        "type": "raw_embedding_table",
        "model_id": "llava-hf/llava-1.5-7b-hf",
        "output_prefix": "llava15_raw",
    },
}


def load_vlm_embedding_table(vlm_choice):
    cfg = VLM_REGISTRY[vlm_choice]
    model_id = cfg["model_id"]

    tokenizer = AutoTokenizer.from_pretrained(
        model_id,
        use_fast=True,
    )

    index_path = hf_hub_download(
        repo_id=model_id,
        filename="model.safetensors.index.json",
    )

    import json

    with open(index_path, "r", encoding="utf-8") as handle:
        index = json.load(handle)

    embedding_keys = [
        key
        for key in index["weight_map"]
        if key.endswith("embed_tokens.weight")
    ]

    assert len(embedding_keys) == 1, embedding_keys

    embedding_key = embedding_keys[0]
    embedding_shard = index["weight_map"][embedding_key]

    shard_path = hf_hub_download(
        repo_id=model_id,
        filename=embedding_shard,
    )

    with safe_open(
        shard_path,
        framework="pt",
        device="cpu",
    ) as handle:
        embedding_weight = (
            handle.get_tensor(embedding_key)
            .contiguous()
        )

    return (
        embedding_weight,
        tokenizer,
        cfg,
        embedding_key,
        embedding_shard,
    )


(
    embedding_weight,
    tokenizer,
    vlm_cfg,
    embedding_key,
    embedding_shard,
) = load_vlm_embedding_table(VLM_CHOICE)

print("Loaded:", vlm_cfg["model_id"])
print("Representation: raw input embedding lookup")
print("Embedding key:", embedding_key)
print("Embedding shard:", embedding_shard)
print("Embedding table:", embedding_weight.shape, embedding_weight.dtype)

In [ ]:
import gc
import hashlib
import json
import time
from pathlib import Path

import torch
import torch.nn.functional as F
from tqdm.auto import tqdm
from IPython.display import FileLink, display


def find_one(filename):
    matches = list(
        Path("/kaggle/input").rglob(filename)
    )

    assert len(matches) == 1, (
        f"Expected one {filename!r}, found: {matches}"
    )

    return matches[0]


QWEN_PATH = find_one(
    "qwen3_tts_minimax_english_100_raw_text_embeddings.pt"
)


qwen = torch.load(
    QWEN_PATH,
    map_location="cpu",
    weights_only=False,
)


required_keys = {
    "embeddings",
    "row_offsets",
    "token_ids",
    "character_offsets",
    "token_strings",
    "prompt_texts",
    "prompt_sha256",
    "wrapped_texts",
    "full_input_ids",
    "source_voices",
    "model_id",
    "target_representation",
}

missing = required_keys - set(qwen)

assert not missing, (
    f"Missing Qwen keys: {missing}"
)


prompts = list(qwen["prompt_texts"])

# There is no SANA CHI prefix here. LLaVA receives exactly the raw TTS text.
conditioned_texts = list(prompts)
chi_prompt = ""

expected_ids = [
    hashlib.sha256(
        prompt.encode("utf-8")
    ).hexdigest()
    for prompt in prompts
]

assert expected_ids == list(qwen["prompt_sha256"])


row_offsets = qwen["row_offsets"].long()
token_counts = row_offsets[1:] - row_offsets[:-1]
sequence_length = int(token_counts.max())
target_dimension = int(qwen["embeddings"].shape[1])


# Reconstruct the same dense target layout used by the source notebook.
target_sequences = torch.zeros(
    len(prompts),
    sequence_length,
    target_dimension,
    dtype=qwen["embeddings"].dtype,
)

target_mask = torch.zeros(
    len(prompts),
    sequence_length,
    dtype=torch.bool,
)

target_ids = torch.zeros(
    len(prompts),
    sequence_length,
    dtype=torch.int32,
)

target_special_mask = torch.zeros(
    len(prompts),
    sequence_length,
    dtype=torch.bool,
)

target_offsets = torch.zeros(
    len(prompts),
    sequence_length,
    2,
    dtype=torch.int32,
)


for prompt_index in range(len(prompts)):
    flat_start = int(row_offsets[prompt_index])
    flat_end = int(row_offsets[prompt_index + 1])
    count = flat_end - flat_start

    target_sequences[prompt_index, :count] = (
        qwen["embeddings"][flat_start:flat_end]
    )

    target_mask[prompt_index, :count] = True

    target_ids[prompt_index, :count] = (
        qwen["token_ids"][flat_start:flat_end]
    )

    target_offsets[prompt_index, :count] = (
        qwen["character_offsets"][flat_start:flat_end]
    )


print("Qwen file:", QWEN_PATH)
print("Fitting prompts:", len(prompts))
print("Fitting target:", target_sequences.shape)
print("Fitting active tokens:", int(target_mask.sum()))
print("Source representation: raw LLaVA embedding lookup")
print("Target representation:", qwen["target_representation"])

In [ ]:
assert VLM_CHOICE == "llava", (
    "Set VLM_CHOICE = 'llava' in Cell 1."
)

tokenizer.padding_side = "right"

if tokenizer.pad_token_id is None:
    tokenizer.pad_token = (
        tokenizer.eos_token
        or tokenizer.unk_token
    )

assert tokenizer.is_fast

SOURCE_BATCH_SIZE = 32


@torch.inference_mode()
def extract_exact_pairs():
    x_parts = []
    y_parts = []
    prompt_index_parts = []
    target_position_parts = []
    fallback_count = 0

    for batch_start in tqdm(
        range(
            0,
            len(conditioned_texts),
            SOURCE_BATCH_SIZE,
        ),
        desc="Looking up and aligning raw embeddings",
    ):
        batch_end = min(
            batch_start + SOURCE_BATCH_SIZE,
            len(conditioned_texts),
        )

        batch_texts = conditioned_texts[
            batch_start:batch_end
        ]

        encoded = tokenizer(
            batch_texts,
            add_special_tokens=True,
            padding=True,
            truncation=False,
            return_offsets_mapping=True,
            return_special_tokens_mask=True,
            return_tensors="pt",
        )

        source_offsets = encoded[
            "offset_mapping"
        ]

        source_special = encoded[
            "special_tokens_mask"
        ].bool()

        source_attention = encoded[
            "attention_mask"
        ].bool()

        source_ids = encoded[
            "input_ids"
        ].long()

        # This is the only source-model operation. There is no transformer
        # model and no forward pass: token IDs directly index the raw table.
        source_embeddings = embedding_weight[
            source_ids
        ]

        for local_index in range(
            len(batch_texts)
        ):
            row_index = (
                batch_start + local_index
            )

            target_positions = torch.nonzero(
                target_mask[row_index],
                as_tuple=False,
            ).flatten()

            target_is_special = (
                target_special_mask[
                    row_index,
                    target_positions,
                ]
            )

            assert not target_is_special.any()

            source_regular_indices = (
                torch.nonzero(
                    source_attention[local_index]
                    & ~source_special[local_index]
                    & (
                        source_offsets[
                            local_index, :, 1
                        ]
                        > source_offsets[
                            local_index, :, 0
                        ]
                    ),
                    as_tuple=False,
                ).flatten()
            )

            target_spans = target_offsets[
                row_index,
                target_positions,
            ]

            source_spans = source_offsets[
                local_index,
                source_regular_indices,
            ]

            overlap_start = torch.maximum(
                target_spans[:, None, 0],
                source_spans[None, :, 0],
            )

            overlap_end = torch.minimum(
                target_spans[:, None, 1],
                source_spans[None, :, 1],
            )

            overlap = (
                overlap_end - overlap_start
            ).clamp_min(0).float()

            missing_overlap = (
                overlap.sum(dim=1) == 0
            )

            if missing_overlap.any():
                target_centers = (
                    target_spans[:, 0]
                    + target_spans[:, 1]
                ).float() / 2

                source_centers = (
                    source_spans[:, 0]
                    + source_spans[:, 1]
                ).float() / 2

                missing_rows = torch.nonzero(
                    missing_overlap,
                    as_tuple=False,
                ).flatten()

                for missing_row in missing_rows:
                    nearest = torch.argmin(
                        torch.abs(
                            source_centers
                            - target_centers[
                                missing_row
                            ]
                        )
                    )

                    overlap[
                        missing_row,
                        nearest,
                    ] = 1

                    fallback_count += 1

            weights = overlap / overlap.sum(
                dim=1,
                keepdim=True,
            ).clamp_min(1)

            source_features = source_embeddings[
                local_index,
                source_regular_indices,
            ].float()

            aligned = (
                weights @ source_features
            )

            targets = target_sequences[
                row_index,
                target_positions,
            ].float()

            x_parts.append(
                aligned.to(torch.float16)
            )

            y_parts.append(
                targets.cpu()
            )

            prompt_index_parts.extend(
                [row_index]
                * len(target_positions)
            )

            target_position_parts.extend(
                target_positions.tolist()
            )

    return (
        torch.cat(x_parts).contiguous(),
        torch.cat(y_parts).contiguous(),
        torch.tensor(
            prompt_index_parts,
            dtype=torch.int32,
        ),
        torch.tensor(
            target_position_parts,
            dtype=torch.int16,
        ),
        fallback_count,
    )


start_time = time.perf_counter()

(
    X_cpu,
    Y_cpu,
    pair_prompt_indices,
    pair_target_positions,
    alignment_fallback_count,
) = extract_exact_pairs()


PAIR_PATH = Path(
    "/kaggle/working/"
    "qwen3_tts_llava_raw_minimax100_fit_pairs.pt"
)

pair_bundle = {
    "format_version": 1,
    "kind": "qwen3_tts_llava_raw_fit_token_pairs",
    "X": X_cpu,
    "Y": Y_cpu,
    "prompt_indices": pair_prompt_indices,
    "target_positions": pair_target_positions,
    "prompt_texts": prompts,
    "conditioned_texts": conditioned_texts,
    "ids": expected_ids,
    "attention_mask": target_mask.to(torch.uint8),
    "target_input_ids": target_ids,
    "target_offsets": target_offsets,
    "chi_prompt": chi_prompt,
    "source_model": vlm_cfg["model_id"],
    "target_model": qwen["model_id"],
    "source_representation": "raw_input_embedding_lookup",
    "target_representation": qwen["target_representation"],
    "source_add_special_tokens": True,
    "source_tokenizer_padding_side": "right",
    "alignment": "character_overlap_weighted",
    "alignment_fallback_count": alignment_fallback_count,
}

torch.save(
    pair_bundle,
    PAIR_PATH,
)

elapsed = time.perf_counter() - start_time

print("Pairs:", X_cpu.shape, Y_cpu.shape)
print("Alignment fallbacks:", alignment_fallback_count)
print("Extraction seconds:", round(elapsed, 2))
print("Saved:", PAIR_PATH)

display(FileLink(str(PAIR_PATH)))

prepare embeds

In [ ]:
# -------------------------
# Prepare embeddings exactly like old calculatingw
# -------------------------

globals().pop("embedding_weight", None)

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

fit_device = torch.device(
    "cuda:0" if torch.cuda.is_available() else "cpu"
)

X_raw = X_cpu.float().to(fit_device)
Y_raw = Y_cpu.float().to(fit_device)


def normalize_rows(T, eps=1e-12):
    return T / T.norm(
        dim=1,
        keepdim=True
    ).clamp_min(eps)


# -------------------------
# Cholesky whitening helpers
# EXACTLY the old approach
# -------------------------

def fit_cholesky_whitener(T, eps=1e-3):

    T = T.float()

    mean = T.mean(dim=0, keepdim=True)
    T_centered = T - mean

    n, d = T_centered.shape

    cov = (
        T_centered.T @ T_centered
    ) / max(n - 1, 1)

    cov = cov + eps * torch.eye(
        d,
        device=T.device,
        dtype=T.dtype
    )

    L = torch.linalg.cholesky(cov)

    T_white = torch.linalg.solve_triangular(
        L,
        T_centered.T,
        upper=False
    ).T

    state = {
        "mean": mean,
        "L": L,
    }

    return T_white, state


def apply_cholesky_whitener(T, state):

    mean = state["mean"]
    L = state["L"]

    T_centered = T - mean

    return torch.linalg.solve_triangular(
        L,
        T_centered.T,
        upper=False
    ).T


def invert_cholesky_whitener(T_white, state):

    mean = state["mean"]
    L = state["L"]

    return T_white @ L.T + mean


# ==========================================================
# X: OLD PIPELINE
# normalize -> center -> normalize
# ==========================================================

X_norm = normalize_rows(X_raw)

X_norm_mean = X_norm.mean(
    dim=0,
    keepdim=True
)

X_norm = X_norm - X_norm_mean

X_norm = normalize_rows(X_norm)


# ==========================================================
# Y: OLD PIPELINE
# normalize -> Cholesky whitening
# ==========================================================

Y_runtime = normalize_rows(Y_raw)

Y_norm, Y_chol_state = fit_cholesky_whitener(
    Y_runtime,
    eps=1e-3
)


print("Using device:", fit_device)
print("X:", X_raw.shape, "->", X_norm.shape)
print("Y:", Y_raw.shape, "->", Y_norm.shape)

Metrics

In [ ]:
def print_loss(proj, target, batch_size=256):
    """
    Old calculatingw-style metrics.
    Uses batching so token-level evaluation does not allocate the full NxN matrix.
    """
    proj = normalize_rows(proj)
    target = normalize_rows(target)

    assert proj.shape == target.shape
    row_count = proj.shape[0]

    correct_values = F.cosine_similarity(proj, target, dim=1)

    wrong_sum = 0.0
    wrong_sq_sum = 0.0
    wrong_count = 0
    top1_correct = 0

    similarity_sum = 0.0
    similarity_max = -float("inf")
    similarity_min = float("inf")

    for start in range(0, row_count, batch_size):
        end = min(start + batch_size, row_count)

        S = proj[start:end] @ target.T

        local_rows = torch.arange(end - start, device=S.device)
        correct_cols = torch.arange(start, end, device=S.device)
        diag = S[local_rows, correct_cols]

        total_sum = S.sum()
        total_sq_sum = (S * S).sum()

        wrong_sum += float(total_sum - diag.sum())
        wrong_sq_sum += float(total_sq_sum - (diag * diag).sum())
        wrong_count += S.numel() - len(diag)

        top1_correct += int(
            (S.argmax(dim=1) == correct_cols).sum()
        )

        similarity_sum += float(total_sum)
        similarity_max = max(similarity_max, float(S.max()))
        similarity_min = min(similarity_min, float(S.min()))

    correct_mean = float(correct_values.mean())
    wrong_mean = wrong_sum / wrong_count
    top1 = top1_correct / row_count

    wrong_variance = max(
        wrong_sq_sum / wrong_count - wrong_mean ** 2,
        0.0,
    )
    wrong_std = wrong_variance ** 0.5

    mse = F.mse_loss(proj, target)
    l2_per_row = torch.norm(proj - target, p=2, dim=1)
    mean_l2 = float(l2_per_row.mean())
    normalized_l2_error = float((l2_per_row / 2).mean())
    l2_closeness = 1 - normalized_l2_error

    print("Correct similarity mean:", correct_mean)
    print("Wrong similarity mean:", wrong_mean)
    print("Top1 accuracy:", top1)
    print("Separation gap:", correct_mean - wrong_mean)
    print(
        "S.mean, max, min:",
        similarity_sum / (row_count * row_count),
        similarity_max,
        similarity_min,
    )
    print("correct std:", float(correct_values.std()))
    print("wrong std:", wrong_std)
    print("MSE:", float(mse))
    print("Mean L2 per row [0,2]:", mean_l2)
    print("Normalized L2 error [0,1]:", normalized_l2_error)
    print("L2 closeness [0,1]:", l2_closeness)
    print("Mean cosine similarity:", correct_mean)
    print("Min cosine similarity:", float(correct_values.min()))
    print("Max cosine similarity:", float(correct_values.max()))

    return {
        "correct_cosine": correct_mean,
        "wrong_cosine": wrong_mean,
        "top1": top1,
        "separation_gap": correct_mean - wrong_mean,
        "mse": float(mse),
        "mean_l2": mean_l2,
        "normalized_l2_error": normalized_l2_error,
        "l2_closeness": l2_closeness,
        "minimum_cosine": float(correct_values.min()),
        "maximum_cosine": float(correct_values.max()),
    }

Krr

In [ ]:
def polynomial_kernel(
    X1,
    X2,
    sigma=0.5003,
    degree=2,
    coef0=0.2
):
    return (
        sigma * (X1 @ X2.T) + coef0
    ) ** degree


sigma = 0.5003
degree = 2
coef0 = 0.2
requested_lambda = 1e-5

fit_start = time.perf_counter()


K = polynomial_kernel(
    X_norm,
    X_norm,
    sigma=sigma,
    degree=degree,
    coef0=coef0,
)

K = (K + K.T) * 0.5

I = torch.eye(
    len(K),
    device=fit_device,
    dtype=K.dtype
)


# IMPORTANT:
# fit directly to old whitened Y
rhs = Y_norm


kernel_cholesky = None
lambda_reg = None

for multiplier in [
    1, 3, 10, 30, 100, 300, 1000
]:

    candidate_lambda = (
        requested_lambda * multiplier
    )

    candidate, info = (
        torch.linalg.cholesky_ex(
            K + candidate_lambda * I
        )
    )

    if int(info.max()) == 0:
        kernel_cholesky = candidate
        lambda_reg = candidate_lambda
        break


assert kernel_cholesky is not None


# This is now the ONLY coefficient matrix.
A = torch.cholesky_solve(
    rhs,
    kernel_cholesky
)


fit_seconds = (
    time.perf_counter() - fit_start
)


print("K:", K.shape)
print("A:", A.shape)
print("Lambda:", lambda_reg)
print(
    "Fit seconds:",
    round(fit_seconds, 2)
)

project and eval

In [ ]:
def transform_source(X):

    X = normalize_rows(X)

    # IMPORTANT:
    # same mean fitted on training X
    X = X - X_norm_mean

    return normalize_rows(X)


@torch.inference_mode()
def reconstruct_prediction(kernel_rows):

    # Predict in whitened Y space
    predicted_white = kernel_rows @ A

    # Undo the SAME whitening used during training
    predicted_runtime = (
        invert_cholesky_whitener(
            predicted_white,
            Y_chol_state
        )
    )

    # Old Y_runtime was row-normalized.
    return normalize_rows(
        predicted_runtime
    )


@torch.inference_mode()
def project_source_rows(
    source_rows,
    batch_size=256
):

    output_parts = []

    for start in tqdm(
        range(
            0,
            len(source_rows),
            batch_size
        ),
        desc="Projecting",
    ):

        end = min(
            start + batch_size,
            len(source_rows)
        )

        source = (
            source_rows[start:end]
            .float()
            .to(fit_device)
        )

        source = transform_source(source)

        K_test = polynomial_kernel(
            source,
            X_norm,
            sigma=sigma,
            degree=degree,
            coef0=coef0,
        )

        output_parts.append(
            reconstruct_prediction(
                K_test
            ).cpu()
        )

    return torch.cat(
        output_parts
    ).contiguous()


# ==========================================
# TRAIN / SAME 100 PROMPTS
# ==========================================

Y_pred = reconstruct_prediction(K)

print(
    "TRAIN / SAME 100 MINIMAX PROMPTS"
)

train_metrics = print_loss(
    Y_pred,
    Y_runtime
)

project scale

In [ ]:
def scale_summary(name, tensor):
    norms = tensor.float().norm(dim=1)

    print(
        f"{name:30s}",
        f"min={norms.min().item():.6f}",
        f"median={norms.median().item():.6f}",
        f"mean={norms.mean().item():.6f}",
        f"max={norms.max().item():.6f}",
    )


print("CURRENT SCALE")
print("=============")

scale_summary("Fitting native targets", Y_raw)
scale_summary("Fitting predictions", Y_pred)

In [ ]:
# ============================================================
# SEPARATE TOKEN-SCALE PROJECTION
#
# Direction projection remains completely unchanged.
# This fits one additional scalar log-norm per token.
# ============================================================

scale_target_norm = Y_raw.norm(
    dim=1,
    keepdim=True,
).clamp_min(1e-12)

scale_target_log = scale_target_norm.log()

scale_log_mean = scale_target_log.mean(
    dim=0,
    keepdim=True,
)

scale_log_std = scale_target_log.std(
    dim=0,
    keepdim=True,
).clamp_min(1e-6)

scale_target_z = (
    scale_target_log - scale_log_mean
) / scale_log_std

scale_min = scale_target_norm.min().item()
scale_max = scale_target_norm.max().item()


# Separate scalar KRR head.
# It reuses the existing kernel decomposition but does not
# modify the direction coefficients A.
scale_coefficients = torch.cholesky_solve(
    scale_target_z,
    kernel_cholesky,
)


@torch.inference_mode()
def predict_norm_from_kernel(kernel_rows):
    predicted_z = (
        kernel_rows @ scale_coefficients
    )

    predicted_log_norm = (
        predicted_z * scale_log_std
        + scale_log_mean
    )

    return (
        predicted_log_norm.exp()
        .clamp(
            min=scale_min,
            max=scale_max,
        )
    )


train_predicted_scale = predict_norm_from_kernel(K)

Y_pred_scaled = (
    normalize_rows(Y_pred.to(fit_device))
    * train_predicted_scale
)

print("Scale coefficients:", scale_coefficients.shape)
print("Training scale range:", scale_min, scale_max)

In [ ]:
print("SCALE AFTER CORRECTION")
print("======================")

scale_summary("Fitting native targets", Y_raw)
scale_summary("Fitting before scale", Y_pred)
scale_summary("Fitting after scale", Y_pred_scaled)


train_native_norm = Y_raw.norm(dim=1)
train_scaled_norm = Y_pred_scaled.norm(dim=1)

train_scale_mae = (
    train_scaled_norm - train_native_norm
).abs().mean().item()

train_scale_correlation = torch.corrcoef(
    torch.stack([
        train_scaled_norm.float(),
        train_native_norm.float(),
    ])
)[0, 1].item()

final_cosines = F.cosine_similarity(
    Y_pred_scaled,
    Y_raw,
    dim=1,
)

print()
print("Training scale MAE:", train_scale_mae)
print("Training scale correlation:", train_scale_correlation)
print("Final mean cosine:", final_cosines.mean().item())
print("Final minimum cosine:", final_cosines.min().item())

assert torch.isfinite(
    Y_pred_scaled
).all()

Save projector and projected Qwen raw embeddings

In [ ]:
import os
from pathlib import Path
from IPython.display import FileLink, display


PROJECTOR_PATH = Path(
    "/kaggle/working/"
    "qwen3_tts_llava_raw_minimax100_projector_token_scale.pt"
)

PROJECTED_PATH = Path(
    "/kaggle/working/"
    "qwen3_tts_minimax_english_100_projected_raw_text_embeddings.pt"
)


projector_bundle = {
    "format_version": 1,
    "kind": "qwen3_tts_llava_raw_token_kernel_projector",
    "source_model": vlm_cfg["model_id"],
    "target_model": qwen["model_id"],
    "source_representation": "raw_input_embedding_lookup",
    "target_representation": qwen["target_representation"],
    "source_add_special_tokens": True,
    "source_tokenizer_padding_side": "right",
    "prompt_texts": prompts,
    "ids": expected_ids,
    "chi_prompt": chi_prompt,
    "prompt_indices": pair_prompt_indices,
    "target_positions": pair_target_positions,

    # X preprocessing
    "X_norm_mean": X_norm_mean.cpu(),
    "train_features": X_norm.cpu(),

    "kernel": {
        "kind": "polynomial",
        "sigma": sigma,
        "degree": degree,
        "coef0": coef0,
        "lambda_reg": lambda_reg,
    },

    # Direction KRR coefficients
    "coefficients": A.cpu(),

    # Y whitening
    "Y_mean": Y_chol_state["mean"].cpu(),
    "Y_cholesky": Y_chol_state["L"].cpu(),
    "Y_cholesky_epsilon": 1e-3,

    # Scale KRR coefficients
    "norm_coefficients": scale_coefficients.cpu(),
    "Y_log_norm_mean": scale_log_mean.cpu(),
    "Y_log_norm_std": scale_log_std.cpu(),
    "Y_norm_min": scale_min,
    "Y_norm_max": scale_max,
    "scale_projection": "kernel_ridge_log_l2_norm",

    "training_metrics": train_metrics,
    "scale_training_mae": train_scale_mae,
    "scale_training_correlation": train_scale_correlation,
    "final_mean_cosine": final_cosines.mean().item(),
    "final_minimum_cosine": final_cosines.min().item(),
    "fit_seconds": fit_seconds,
    "benchmark_prompts_used_for_fitting": True,
}


projected_bundle = {
    "format_version": 2,
    "kind": "qwen3_tts_projected_raw_text_embeddings",
    "projected_text_mode": "native_qwen_content_token_sequence",
    "embeddings": Y_pred_scaled.cpu().to(torch.float16).contiguous(),
    "native_target_embeddings": qwen["embeddings"].clone(),
    "conditioning_storage": "active_content_tokens_only",
    "row_offsets": qwen["row_offsets"].clone(),
    "token_ids": qwen["token_ids"].clone(),
    "character_offsets": qwen["character_offsets"].clone(),
    "token_strings": list(qwen["token_strings"]),
    "prompt_texts": list(qwen["prompt_texts"]),
    "prompt_sha256": list(qwen["prompt_sha256"]),
    "source_voices": list(qwen["source_voices"]),
    "wrapped_texts": list(qwen["wrapped_texts"]),
    "full_input_ids": list(qwen["full_input_ids"]),
    "source_model": vlm_cfg["model_id"],
    "target_model": qwen["model_id"],
    "source_representation": "raw_input_embedding_lookup",
    "target_representation": qwen["target_representation"],
    "injection_point": "immediately_before_talker.text_projection",
    "conditioning_layout": "exact_native_qwen_content_positions",
    "projector_kind": projector_bundle["kind"],
    "projector_training_metrics": train_metrics,
    "scale_projection": "kernel_ridge_log_l2_norm",
    "scale_training_mae": train_scale_mae,
    "scale_training_correlation": train_scale_correlation,
    "benchmark_prompts_used_for_fitting": True,
    "evaluation_validity": "same_data_pipeline_sanity_check",
}


assert projected_bundle["embeddings"].shape == qwen["embeddings"].shape
assert torch.equal(
    projected_bundle["row_offsets"],
    qwen["row_offsets"],
)


torch.save(
    projector_bundle,
    PROJECTOR_PATH,
)

torch.save(
    projected_bundle,
    PROJECTED_PATH,
)


saved_projected = torch.load(
    PROJECTED_PATH,
    map_location="cpu",
    weights_only=False,
)

assert torch.equal(
    saved_projected["row_offsets"],
    qwen["row_offsets"],
)

assert saved_projected["prompt_sha256"] == qwen["prompt_sha256"]
assert torch.isfinite(saved_projected["embeddings"].float()).all()


print("Saved projector:", PROJECTOR_PATH)
print("Saved projected embeddings:", PROJECTED_PATH)
print("Projected shape:", saved_projected["embeddings"].shape)

scale_summary(
    "Saved projected artifact",
    saved_projected["embeddings"],
)

os.chdir("/kaggle/working")
display(FileLink(PROJECTOR_PATH.name))
display(FileLink(PROJECTED_PATH.name))